
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lecture — Data Imputation

## Overview

Missing values are one of the first problems you meet when preparing ML data. This lecture shows how to find and handle them **on Databricks**: built-in tools show you where values are missing, and Spark ML's `Imputer` fills them in with values learned from the training data, so that training and inference handle missing data the same way.

This lecture covers the following sections:

- **A. Find Missing Values on Databricks:** the Data Profile, `summary()`, null counts with Spark, Genie Code, and Data profiling in Unity Catalog
- **B. Choose a Strategy:** drop, treat missing as a category, use a domain default, or impute, with the Spark code for each
- **C. Impute with Spark ML:** the `Imputer` estimator, missingness indicators, and a reusable pipeline fit on training data only
- **D. Optional: Model-Based Imputation:** KNN and regression imputation, and when they are worth the extra cost
- **E. Best Practices:** choosing a method and keeping track of imputed values

## Learning Objectives

By the end of this lecture, you will be able to:

- Find missing values in a Spark DataFrame with Databricks tools: the Data Profile, `summary()`, Spark null counts, Genie Code, and Data profiling in Unity Catalog
- Explain how the pattern of missing values can bias a model
- Choose between dropping, treating missing as a category, using a domain default, and imputing
- Explain how Spark ML's `Imputer` learns mean, median, or mode values from training data and applies them consistently to new data
- Explain when to add a missingness indicator, and why it is created before imputing
- Describe when model-based imputation (KNN or regression) is worth its extra cost

## A. Find Missing Values on Databricks

### A1. Find Missing Values on Databricks

**Data imputation** means filling in missing values with estimated values. Before you choose how to handle missing data, find out **where** it is missing and **how much**. Databricks gives you several ways to do this without writing much code:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Tool</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">How to use it</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Best for</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Data Profile</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Run <code>display(df)</code>, then select <b>+</b> &gt; <b>Data Profile</b> above the results</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A quick visual summary of every column, including the percentage of missing values</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;"><code>summary()</code></span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><code>display(df.summary("count"))</code> shows the number of non-null values in each column</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A fast numeric check in code</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Null counts with Spark</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Count <code>isNull()</code> per column (see below)</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">An exact count you can use in later code, for example to choose columns to drop</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #7B61FF; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F3F0FF; color: #7B61FF; font-weight: 700; border: 1px solid #7B61FF40; white-space: nowrap;">Genie Code</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Ask: <i>"Show the percentage of missing values in each column of telco_df, sorted from highest to lowest"</i></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Generating the analysis code for you; always review the code before you run it</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Data profiling in Unity Catalog</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Catalog Explorer &gt; table &gt; <b>Quality</b> tab</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Tracking missing values on a schedule for tables that refresh, as covered in Module 1</td></tr></tbody></table></div>

To count the missing values in every column with Spark:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.sql import functions as F</code>
</br>
<code># Count the missing values in every column</code>
<code>missing_counts = telco_df.select([</code>
<code>    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in telco_df.columns</code>
<code>])</code>
<code>display(missing_counts)</code>
</br>
<code># The &quot;count&quot; row of summary() is the number of non-null values per column</code>
<code>display(telco_df.summary(&quot;count&quot;))</code></pre>
</div>

These checks run in Spark, so they work the same way on a sample or on the full table.

### A2. Why the Pattern of Missing Values Matters

Missing values can reduce model quality, introduce bias, or both. How much they matter depends on **why** the values are missing, **how many** are missing, and how you handle them.

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Problem</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Why it hurts the model</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Errors and lost rows</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Spark ML's <code>VectorAssembler</code> raises an error on null inputs by default, and dropping every incomplete row can remove a large part of the data</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Biased results</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">If missingness is related to the target (for example, customers who are about to leave skip a survey), dropping or naively filling those rows skews the model</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Under-represented groups</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Groups with more missing data become under-represented after rows are dropped</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FF5F46; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF1EE; color: #FF5F46; font-weight: 700; border: 1px solid #FF5F4640; white-space: nowrap;">Training/inference mismatch</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">If missing values are handled one way in training and another way in production, the model sees different data. A fitted Spark ML pipeline applies the same handling everywhere</td></tr></tbody></table></div>

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Caution</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Why a value is missing is usually an assumption, not something you can prove from the data alone. Compare missing-value rates across important groups (for example, by class or by region) and use domain knowledge before you decide how to handle them.</div></div>

## B. Choose a Strategy

There are four common ways to handle missing data. Most projects use a mix, chosen column by column.

<div class="cw">
<style>
.cw { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 4px 16px rgba(27,49,57,0.13); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 14pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-text { font-size: 12pt; color: #5E7077; line-height: 1.55; margin: 0; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
.tc { text-align: center; }
</style>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-title">1. Drop</div>
    <ul>
      <li>Drop a column that is mostly empty: <code>df.drop()</code></li>
      <li>Drop rows with too few values: <code>df.na.drop(thresh=k)</code></li>
      <li>Risk: bias if the values are not missing at random</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-title">2. Treat as a category</div>
    <ul>
      <li>Fill with <code>"Unknown"</code>: <code>df.fillna()</code></li>
      <li>Or keep nulls: <code>StringIndexer</code> with <code>handleInvalid="keep"</code></li>
      <li>Best for categorical columns</li>
    </ul>
  </div>
  <div class="ds-card amber">
    <div class="ds-card-title">3. Domain default</div>
    <ul>
      <li>Fill with a known meaning, such as <code>"No"</code></li>
      <li>Only when the data definition confirms what missing means</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-title">4. Impute</div>
    <ul>
      <li>Spark ML <code>Imputer</code>: mean, median, or mode</li>
      <li>Learned from the training set, applied to all data</li>
      <li>Optional: model-based methods (section D)</li>
    </ul>
  </div>
</div>
</div>

### B1. Drop Columns or Rows

Use this when there is too little information to be worth recovering:

- **Drop a column** if it is missing in most rows and is not important for the model
- **Drop a row** if most of its fields are missing

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>dropped_df = telco_df.drop(&quot;mostly_empty_col&quot;)   # drop a column that is mostly empty and not important</code>
<code>dropped_df = dropped_df.na.drop(thresh=16)       # keep rows with at least 16 non-null values</code></pre>
</div>

`na.drop(thresh=k)` keeps only rows that have **at least `k` non-null values**. There is no universal threshold: decide it from how important the feature is, whether it is available at prediction time, and validation results.

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Dropping rows or columns with missing data can introduce bias if the missingness is not random. Check whether the dropped rows share meaningful characteristics before you remove them.</div></div>

### B2. Treat as a Category, or Use a Domain Default

For **categorical** columns, "missing" can be meaningful, for example a customer who did not select a payment method. You can give missing values their own category, such as `"Unknown"`, so the model can learn from them.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml.feature import StringIndexer</code>
</br>
<code># Option 1: an explicit category for missing values</code>
<code>category_df = telco_df.fillna(&quot;Unknown&quot;, subset=[&quot;PaymentMethod&quot;])</code>
</br>
<code># Option 2: keep the nulls and let StringIndexer give them their own index</code>
<code>indexer = StringIndexer(inputCol=&quot;PaymentMethod&quot;, outputCol=&quot;PaymentMethod_idx&quot;, handleInvalid=&quot;keep&quot;)</code>
</br>
<code># Option 3: a domain default, only when the data definition says missing means &quot;No&quot;</code>
<code>default_df = telco_df.fillna(&quot;No&quot;, subset=[&quot;OnlineSecurity&quot;])</code></pre>
</div>

Missing, "not applicable", and an explicit "No" are different states. Use a default such as `"No"` only when the data definition confirms that missing means "No".

<div style="max-width: 330px; margin: 0 auto;">
<svg width="100%" viewBox="0 20 250 220" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <!-- Before rectangle -->
  <rect x="8" y="30" width="204" height="70" rx="10" fill="#F9F7F4" stroke="#2272B4" stroke-width="1.5"/>
  <text x="110" y="50" text-anchor="middle" font-size="14" font-weight="700" fill="#2272B4">Before</text>
  <text x="110" y="70" text-anchor="middle" font-size="12" fill="#0b2026">PaymentMethod:</text>
  <text x="110" y="90" text-anchor="middle" font-size="12" fill="#618794">"Credit", "Debit", <tspan fill="#FF5F46">null</tspan></text>
  <!-- Arrow -->
  <defs>
    <marker id="arrowhead" markerWidth="8" markerHeight="8" refX="6" refY="4" orient="auto">
      <polygon points="0,0 8,4 0,8" fill="#2272B4"/>
    </marker>
  </defs>
  <line x1="110" y1="100" x2="110" y2="150" stroke="#2272B4" stroke-width="2" marker-end="url(#arrowhead)"/>
  <text x="175" y="125" text-anchor="middle" font-size="11" fill="#2272B4">Categorical Encoding</text>
  <!-- After rectangle -->
  <rect x="8" y="160" width="204" height="70" rx="10" fill="#F9F7F4" stroke="#2272B4" stroke-width="1.5"/>
  <text x="110" y="180" text-anchor="middle" font-size="14" font-weight="700" fill="#2272B4">After</text>
  <text x="110" y="200" text-anchor="middle" font-size="12" fill="#0b2026">PaymentMethod:</text>
  <text x="110" y="220" text-anchor="middle" font-size="12" fill="#618794">"Credit", "Debit", <tspan fill="#FF5F46">"Missing"</tspan></text>
</svg>
</div>

### B3. Replace with an Estimated Value (Impute)

For **numerical** columns, dropping is often too aggressive, and a category label does not capture the underlying value, so you estimate a replacement. On Databricks, you do this with Spark ML's `Imputer`, covered next.

## C. Impute with Spark ML

### C1. The Spark ML `Imputer`: Mean, Median, or Mode

`Imputer` is a Spark ML **estimator**. When you call `fit()`, it computes the replacement value for each column from the data you give it. When you call `transform()`, it fills in the missing values (nulls and `NaN`) with those learned values. Fit it on the **training set only**, then apply the fitted model to the training, test, and future inference data.

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml.feature import Imputer</code>
</br>
<code>train_df, test_df = telco_df.randomSplit([0.8, 0.2], seed=42)</code>
</br>
<code>imputer = Imputer(</code>
<code>    inputCols=[&quot;tenure&quot;, &quot;TotalCharges&quot;],</code>
<code>    outputCols=[&quot;tenure_imputed&quot;, &quot;TotalCharges_imputed&quot;],</code>
<code>    strategy=&quot;median&quot;,              # &quot;mean&quot;, &quot;median&quot;, or &quot;mode&quot;</code>
<code>)</code>
</br>
<code>imputer_model = imputer.fit(train_df)           # learns the medians from the training set only</code>
<code>train_imputed = imputer_model.transform(train_df)</code>
<code>test_imputed = imputer_model.transform(test_df) # reuses the training medians</code>
</br>
<code>display(imputer_model.surrogateDF)              # the values that will be filled in</code></pre>
</div>

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Strategy</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Use for</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Notes</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;"><code>mean</code></span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Numeric columns with a roughly symmetric distribution</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Sensitive to outliers</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;"><code>median</code></span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Numeric columns that are skewed or have outliers</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Robust to outliers; a good default for amounts and counts</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #5A6F77; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #F0F2F4; color: #5A6F77; font-weight: 700; border: 1px solid #5A6F7740; white-space: nowrap;"><code>mode</code></span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Numeric columns with a few discrete values</td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">For string columns, use <code>fillna()</code> or <code>StringIndexer</code> instead; <code>Imputer</code> works on numeric columns only. <b>Not supported on Serverless compute</b> (see the note below)</td></tr></tbody></table></div>

The diagram below shows mean imputation: each observed value contributes to the value that fills the gap.

<div style="max-width: 480px; margin: 16px 0;">
<svg width="100%" viewBox="0 0 480 320" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Mean Imputation Visualization</title>
  <desc>Values 10, 15, 20, 25 contribute to the imputed mean of 17.5</desc>
  <defs>
    <marker id="mean-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#FF5F46" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Header bar -->
  <rect x="20" y="10" width="440" height="36" rx="4" fill="#F9F7F4"/>
  <line x1="20" y1="46" x2="460" y2="46" stroke="#FF5F46" stroke-width="2"/>
  <text x="80" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">ID</text>
  <text x="200" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">Before</text>
  <text x="400" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">After (mean)</text>
  <!-- Row 1: ID=1, value=10 -->
  <rect x="20" y="50" width="440" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="94" x2="460" y2="94" stroke="#EEEDE9" stroke-width="1"/>
  <text x="80" y="76" text-anchor="middle" font-size="13" fill="#1B3139">1</text>
  <text x="200" y="76" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">10</text>
  <text x="400" y="76" text-anchor="middle" font-size="13" fill="#1B3139">10.0</text>
  <!-- Row 2: ID=2, value=15 -->
  <rect x="20" y="94" width="440" height="44" fill="#ffffff"/>
  <line x1="20" y1="138" x2="460" y2="138" stroke="#EEEDE9" stroke-width="1"/>
  <text x="80" y="120" text-anchor="middle" font-size="13" fill="#1B3139">2</text>
  <text x="200" y="120" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">15</text>
  <text x="400" y="120" text-anchor="middle" font-size="13" fill="#1B3139">15.0</text>
  <!-- Row 3: ID=3, missing → 17.5 -->
  <rect x="20" y="138" width="440" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="182" x2="460" y2="182" stroke="#EEEDE9" stroke-width="1"/>
  <text x="80" y="164" text-anchor="middle" font-size="13" fill="#1B3139">3</text>
  <text x="200" y="164" text-anchor="middle" font-size="13" fill="#618794">—</text>
  <!-- Highlighted imputed value -->
  <rect x="360" y="148" width="80" height="28" rx="4" fill="#FF5F46" fill-opacity="0.1" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="400" y="166" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">17.5</text>
  <!-- Row 4: ID=4, value=20 -->
  <rect x="20" y="182" width="440" height="44" fill="#ffffff"/>
  <line x1="20" y1="226" x2="460" y2="226" stroke="#EEEDE9" stroke-width="1"/>
  <text x="80" y="208" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="200" y="208" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">20</text>
  <text x="400" y="208" text-anchor="middle" font-size="13" fill="#1B3139">20.0</text>
  <!-- Row 5: ID=5, value=25 -->
  <rect x="20" y="226" width="440" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="270" x2="460" y2="270" stroke="#EEEDE9" stroke-width="1"/>
  <text x="80" y="252" text-anchor="middle" font-size="13" fill="#1B3139">5</text>
  <text x="200" y="252" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">25</text>
  <text x="400" y="252" text-anchor="middle" font-size="13" fill="#1B3139">25.0</text>
  <!-- Arrows from each source value to the imputed 17.5 -->
  <path d="M220 76 C280 76, 340 130, 360 160" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#mean-arrow)"/>
  <path d="M220 120 C270 120, 320 140, 360 158" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#mean-arrow)"/>
  <path d="M220 208 C270 208, 320 188, 360 168" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#mean-arrow)"/>
  <path d="M220 252 C280 252, 340 200, 360 170" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#mean-arrow)"/>
</svg>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">Note</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Filling every gap with one value keeps the row count but piles values up at that one point and reduces the column's variation. Compare the distribution of the column before and after imputation.
    <br><br><b>On Serverless compute:</b> <code>strategy="mode"</code> fails with a <code>UC_COMMAND_NOT_SUPPORTED</code> error, because Unity Catalog on Serverless does not support the higher-order functions it uses. For a binary 0/1 column, use <code>strategy="median"</code> instead: the median of a 0/1 column is its more common value, so the column stays 0 or 1. The 2.6 Demo uses this approach.</div></div>

### C2. Put Imputation in a Reusable Pipeline

In practice, imputation is one stage of a Spark ML **Pipeline**. You fit the pipeline once on the training data, and every statistic it needs (medians, scaling factors, category lists) is learned at that point. You then save the fitted pipeline and use the same object for test data and for inference, so missing values are handled the same way everywhere.

If the fact that a value was missing might carry signal, add an **indicator column before imputing**, so the original missingness is kept:

<div style="background: #F9F7F4; border: 1px solid #DCE0E2; border-radius: 8px; padding: 14px 18px; margin: 12px 0; overflow-x: auto; box-shadow: 0 2px 8px rgba(27,49,57,0.06);">
<pre style="margin: 0; font-family: ui-monospace, SFMono-Regular, Consolas, 'Liberation Mono', Menlo, monospace; font-size: 13px; color: #0b2026; white-space: pre-wrap;">
<code>from pyspark.ml import Pipeline</code>
<code>from pyspark.ml.feature import Imputer, VectorAssembler</code>
<code>from pyspark.sql import functions as F</code>
</br>
<code>def add_missing_flag(df):</code>
<code>    # Create the indicator BEFORE imputing, so the original missingness is kept</code>
<code>    return df.withColumn(&quot;TotalCharges_was_missing&quot;, F.col(&quot;TotalCharges&quot;).isNull().cast(&quot;int&quot;))</code>
</br>
<code>train_df, test_df = add_missing_flag(train_df), add_missing_flag(test_df)</code>
</br>
<code>pipeline = Pipeline(stages=[</code>
<code>    Imputer(inputCols=[&quot;tenure&quot;, &quot;TotalCharges&quot;],</code>
<code>            outputCols=[&quot;tenure_imputed&quot;, &quot;TotalCharges_imputed&quot;],</code>
<code>            strategy=&quot;median&quot;),</code>
<code>    VectorAssembler(inputCols=[&quot;tenure_imputed&quot;, &quot;TotalCharges_imputed&quot;, &quot;TotalCharges_was_missing&quot;],</code>
<code>                    outputCol=&quot;features&quot;),</code>
<code>])</code>
</br>
<code>pipeline_model = pipeline.fit(train_df)          # every statistic is learned from training data</code>
<code>test_prepared = pipeline_model.transform(test_df)</code>
</br>
<code># Save the fitted pipeline to a Unity Catalog volume so training and inference use the same logic</code>
<code>pipeline_model.write().overwrite().save(&quot;/Volumes/&lt;catalog&gt;/&lt;schema&gt;/&lt;volume&gt;/imputation_pipeline_v1&quot;)</code></pre>
</div>

<div style="border-left: 4px solid #00A972; background: #E6F5EE; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #00805A;">Databricks tip</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Saving the fitted pipeline to a <b>Unity Catalog volume</b> puts it under the same permissions and governance as your data. You build a complete pipeline like this in the next two demos.</div></div>

## D. Optional: Model-Based Imputation

### D1. K-Nearest Neighbors (KNN) Imputation

<div style="border-left: 4px solid #FFAB00; background: #FFF7E6; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #B26A00;">Caution</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">This section is optional background. Spark ML does not include a KNN imputer, and model-based imputation adds a model you must fit on the training set, validate, and maintain. Start with <code>Imputer</code>, and use these methods only when validation shows a clear benefit.</div></div>

For each row with a missing value, find its *k* nearest neighbors using the **other columns**, then impute as the mean of their values in the missing column.

<div style="max-width: 620px; margin: 16px 0;">
<svg width="100%" viewBox="0 0 620 370" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>KNN Imputation Visualization (k=2)</title>
  <desc>MonthlyCharges imputed for rows 3 and 6 using Age and Tenure as distance features</desc>
  <defs>
    <marker id="knn-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#FF5F46" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Header bar -->
  <rect x="20" y="10" width="580" height="36" rx="4" fill="#F9F7F4"/>
  <line x1="20" y1="46" x2="600" y2="46" stroke="#FF5F46" stroke-width="2"/>
  <text x="60" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">ID</text>
  <text x="160" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">Age</text>
  <text x="270" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">Tenure</text>
  <text x="430" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">MonthlyCharges</text>
  <text x="560" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">After</text>
  <!-- Row 1: ID=1, Age=30, Tenure=12, MonthlyCharges=65 -->
  <rect x="20" y="50" width="580" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="94" x2="600" y2="94" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="76" text-anchor="middle" font-size="13" fill="#1B3139">1</text>
  <text x="160" y="76" text-anchor="middle" font-size="13" fill="#1B3139">30</text>
  <text x="270" y="76" text-anchor="middle" font-size="13" fill="#1B3139">12</text>
  <text x="430" y="76" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">65</text>
  <text x="560" y="76" text-anchor="middle" font-size="13" fill="#1B3139">65</text>
  <!-- Row 2: ID=2, Age=32, Tenure=11, MonthlyCharges=70 -->
  <rect x="20" y="94" width="580" height="44" fill="#ffffff"/>
  <line x1="20" y1="138" x2="600" y2="138" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="120" text-anchor="middle" font-size="13" fill="#1B3139">2</text>
  <text x="160" y="120" text-anchor="middle" font-size="13" fill="#1B3139">32</text>
  <text x="270" y="120" text-anchor="middle" font-size="13" fill="#1B3139">11</text>
  <text x="430" y="120" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">70</text>
  <text x="560" y="120" text-anchor="middle" font-size="13" fill="#1B3139">70</text>
  <!-- Row 3: ID=3, Age=31, Tenure=12, MonthlyCharges=— (imputed → 67.5) -->
  <rect x="20" y="138" width="580" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="182" x2="600" y2="182" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="164" text-anchor="middle" font-size="13" fill="#1B3139">3</text>
  <text x="160" y="164" text-anchor="middle" font-size="13" fill="#1B3139">31</text>
  <text x="270" y="164" text-anchor="middle" font-size="13" fill="#1B3139">12</text>
  <text x="430" y="164" text-anchor="middle" font-size="13" fill="#618794">—</text>
  <rect x="520" y="148" width="80" height="28" rx="4" fill="#FF5F46" fill-opacity="0.1" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="560" y="166" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">67.5</text>
  <!-- Row 4: ID=4, Age=45, Tenure=24, MonthlyCharges=85 -->
  <rect x="20" y="182" width="580" height="44" fill="#ffffff"/>
  <line x1="20" y1="226" x2="600" y2="226" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="208" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="160" y="208" text-anchor="middle" font-size="13" fill="#1B3139">45</text>
  <text x="270" y="208" text-anchor="middle" font-size="13" fill="#1B3139">24</text>
  <text x="430" y="208" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">85</text>
  <text x="560" y="208" text-anchor="middle" font-size="13" fill="#1B3139">85</text>
  <!-- Row 5: ID=5, Age=47, Tenure=22, MonthlyCharges=90 -->
  <rect x="20" y="226" width="580" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="270" x2="600" y2="270" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="252" text-anchor="middle" font-size="13" fill="#1B3139">5</text>
  <text x="160" y="252" text-anchor="middle" font-size="13" fill="#1B3139">47</text>
  <text x="270" y="252" text-anchor="middle" font-size="13" fill="#1B3139">22</text>
  <text x="430" y="252" text-anchor="middle" font-size="13" font-weight="600" fill="#2272B4">90</text>
  <text x="560" y="252" text-anchor="middle" font-size="13" fill="#1B3139">90</text>
  <!-- Row 6: ID=6, Age=46, Tenure=23, MonthlyCharges=— (imputed → 87.5) -->
  <rect x="20" y="270" width="580" height="44" fill="#ffffff"/>
  <line x1="20" y1="314" x2="600" y2="314" stroke="#EEEDE9" stroke-width="1"/>
  <text x="60" y="296" text-anchor="middle" font-size="13" fill="#1B3139">6</text>
  <text x="160" y="296" text-anchor="middle" font-size="13" fill="#1B3139">46</text>
  <text x="270" y="296" text-anchor="middle" font-size="13" fill="#1B3139">23</text>
  <text x="430" y="296" text-anchor="middle" font-size="13" fill="#618794">—</text>
  <rect x="520" y="280" width="80" height="28" rx="4" fill="#FF5F46" fill-opacity="0.1" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="560" y="298" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">87.5</text>
  <!-- Arrows: row 1 (65) and row 2 (70) → row 3 imputed 67.5 -->
  <path d="M460 76 C490 76, 510 120, 525 152" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#knn-arrow)"/>
  <path d="M460 120 C490 120, 510 140, 525 154" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#knn-arrow)"/>
  <!-- Arrows: row 4 (85) and row 5 (90) → row 6 imputed 87.5 -->
  <path d="M460 208 C490 208, 510 250, 525 284" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#knn-arrow)"/>
  <path d="M460 252 C490 252, 510 270, 525 286" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#knn-arrow)"/>
  <!-- Label: distance features -->
  <rect x="120" y="330" width="200" height="26" rx="4" fill="#EBF3FA" stroke="#2272B4" stroke-width="1"/>
  <text x="220" y="347" text-anchor="middle" font-size="11" font-weight="600" fill="#2272B4">↑ Distance features (Age, Tenure)</text>
  <!-- Label: imputed column -->
  <rect x="380" y="330" width="220" height="26" rx="4" fill="#FFF1EE" stroke="#FF5F46" stroke-width="1"/>
  <text x="490" y="347" text-anchor="middle" font-size="11" font-weight="600" fill="#FF5F46">↑ Imputed column (MonthlyCharges)</text>
</svg>
</div>

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">On Databricks</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">Spark ML has no built-in KNN imputer. Libraries such as scikit-learn's <code>KNNImputer</code> run on a single machine, so you would have to bring the data to the driver with <code>toPandas()</code>. Use them only for data that fits comfortably in memory, and fit them on the training set only.</div></div>

### D2. Regression Imputation

Train a model on the complete rows, then predict the missing value. The model learns relationships between features — so unlike mean imputation, the predicted value respects inter-column structure.

<div style="max-width: 560px; margin: 16px 0;">
<svg width="100%" viewBox="0 0 560 320" role="img" style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <title>Regression Imputation Visualization</title>
  <desc>A linear model trained on complete rows predicts the missing value in row 5</desc>
  <defs>
    <marker id="reg-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="5" markerHeight="5" orient="auto-start-reverse">
      <path d="M2 1L8 5L2 9" fill="none" stroke="#FF5F46" stroke-width="1.8" stroke-linecap="round" stroke-linejoin="round"/>
    </marker>
  </defs>
  <!-- Header bar -->
  <rect x="20" y="10" width="520" height="36" rx="4" fill="#F9F7F4"/>
  <line x1="20" y1="46" x2="540" y2="46" stroke="#FF5F46" stroke-width="2"/>
  <text x="70" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">ID</text>
  <text x="170" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">x₁</text>
  <text x="270" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">x₂</text>
  <text x="390" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">y (observed)</text>
  <text x="500" y="33" text-anchor="middle" font-size="12" font-weight="700" fill="#5A6F77">y (imputed)</text>
  <!-- Row 1: x1=2, x2=4, y=12 -->
  <rect x="20" y="50" width="520" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="94" x2="540" y2="94" stroke="#EEEDE9" stroke-width="1"/>
  <text x="70" y="76" text-anchor="middle" font-size="13" fill="#1B3139">1</text>
  <text x="170" y="76" text-anchor="middle" font-size="13" fill="#1B3139">2</text>
  <text x="270" y="76" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="390" y="76" text-anchor="middle" font-size="13" fill="#1B3139">12</text>
  <text x="500" y="76" text-anchor="middle" font-size="13" fill="#1B3139">12</text>
  <!-- Row 2: x1=3, x2=6, y=17 -->
  <rect x="20" y="94" width="520" height="44" fill="#ffffff"/>
  <line x1="20" y1="138" x2="540" y2="138" stroke="#EEEDE9" stroke-width="1"/>
  <text x="70" y="120" text-anchor="middle" font-size="13" fill="#1B3139">2</text>
  <text x="170" y="120" text-anchor="middle" font-size="13" fill="#1B3139">3</text>
  <text x="270" y="120" text-anchor="middle" font-size="13" fill="#1B3139">6</text>
  <text x="390" y="120" text-anchor="middle" font-size="13" fill="#1B3139">17</text>
  <text x="500" y="120" text-anchor="middle" font-size="13" fill="#1B3139">17</text>
  <!-- Row 3: x1=4, x2=8, y=22 -->
  <rect x="20" y="138" width="520" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="182" x2="540" y2="182" stroke="#EEEDE9" stroke-width="1"/>
  <text x="70" y="164" text-anchor="middle" font-size="13" fill="#1B3139">3</text>
  <text x="170" y="164" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="270" y="164" text-anchor="middle" font-size="13" fill="#1B3139">8</text>
  <text x="390" y="164" text-anchor="middle" font-size="13" fill="#1B3139">22</text>
  <text x="500" y="164" text-anchor="middle" font-size="13" fill="#1B3139">22</text>
  <!-- Row 4: x1=5, x2=2, y=19 -->
  <rect x="20" y="182" width="520" height="44" fill="#ffffff"/>
  <line x1="20" y1="226" x2="540" y2="226" stroke="#EEEDE9" stroke-width="1"/>
  <text x="70" y="208" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="170" y="208" text-anchor="middle" font-size="13" fill="#1B3139">5</text>
  <text x="270" y="208" text-anchor="middle" font-size="13" fill="#1B3139">2</text>
  <text x="390" y="208" text-anchor="middle" font-size="13" fill="#1B3139">19</text>
  <text x="500" y="208" text-anchor="middle" font-size="13" fill="#1B3139">19</text>
  <!-- Row 5: x1=4, x2=5, y=— → imputed 19 -->
  <rect x="20" y="226" width="520" height="44" fill="#F9F7F4"/>
  <line x1="20" y1="270" x2="540" y2="270" stroke="#EEEDE9" stroke-width="1"/>
  <text x="70" y="252" text-anchor="middle" font-size="13" fill="#1B3139">5</text>
  <text x="170" y="252" text-anchor="middle" font-size="13" fill="#1B3139">4</text>
  <text x="270" y="252" text-anchor="middle" font-size="13" fill="#1B3139">5</text>
  <text x="390" y="252" text-anchor="middle" font-size="13" fill="#618794">—</text>
  <rect x="462" y="236" width="76" height="28" rx="4" fill="#FF5F46" fill-opacity="0.1" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="500" y="254" text-anchor="middle" font-size="13" font-weight="700" fill="#FF5F46">19</text>
  <!-- Arrows from complete rows to imputed value -->
  <path d="M420 76 C450 76, 460 180, 468 240" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#reg-arrow)"/>
  <path d="M420 120 C445 120, 455 180, 468 242" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#reg-arrow)"/>
  <path d="M420 164 C445 164, 455 200, 468 244" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#reg-arrow)"/>
  <path d="M420 208 C445 208, 455 220, 468 244" fill="none" stroke="#FF5F46" stroke-width="1.5" stroke-dasharray="4 2" marker-end="url(#reg-arrow)"/>
</svg>
</div>

Learned model from rows 1–4:

$$\hat{y} = 2 + 3x_1 + 1x_2$$

For row 5:

$$\hat{y} = 2 + 3(4) + 1(5) = 19$$

The prediction uses the actual relationships between columns — unlike mean imputation, which would give (12+17+22+19)/4 = 17.5 regardless of what x₁ and x₂ say about row 5.

<div style="border-left: 4px solid #1B5162; background: #F8F9FC; padding: 16px 20px; border-radius: 4px; margin: 16px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><strong style="color: #1B5162;">On Databricks</strong><div style="margin: 8px 0 0 0; color: #0b2026; line-height: 1.6;">You can build regression imputation at scale with any Spark ML regressor: train it on the training rows where the column is present, then predict the missing values. Keep in mind that the prediction is a single best guess. It does not show how uncertain the imputed value is.</div></div>

## E. Best Practices

### E1. Choosing the Right Method

There is no single best approach. The choice depends on several factors:

<div class="ft" style="max-width: 900px; margin: 18px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;"><style>.ft tbody tr { transition: background .15s ease; } .ft tbody tr:hover { background: #F4F8FB; } .ft code { background: #EEEDE9; border-radius: 4px; padding: 1px 5px; }</style><table style="border-collapse: separate; border-spacing: 0; width: 100%; background: #ffffff; border: 1px solid #E4E9F2; border-radius: 12px; overflow: hidden; box-shadow: 0 4px 16px rgba(27,49,57,0.08); font-size: 14.5px;"><thead><tr><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Factor</th><th style="padding: 12px 16px; text-align: left; font-size: 12px; font-weight: 700; letter-spacing: 0.08em; text-transform: uppercase; color: #5A6F77; background: #F9F7F4; border-bottom: 2px solid #FF5F46;">Implication</th></tr></thead><tbody><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Data type</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Categorical → <code>"Unknown"</code> category, domain default, or <code>StringIndexer(handleInvalid="keep")</code>; numeric → <code>Imputer</code> with mean or median</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #4299E0; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EEF6FD; color: #4299E0; font-weight: 700; border: 1px solid #4299E040; white-space: nowrap;">Distribution</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">Skewed or with outliers → prefer median over mean</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Amount missing</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">There is no universal threshold. A high missing rate does not automatically call for a model-based method; consider how important the feature is and what validation shows</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #FFAB00; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #FFF7E6; color: #B26A00; font-weight: 700; border: 1px solid #FFAB0040; white-space: nowrap;">Missingness pattern</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">If values may be missing for a reason, consider an indicator column and check whether it helps</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #00A972; vertical-align: top; border-bottom: 1px solid #EEEDE9;"><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #E6F5EE; color: #00A972; font-weight: 700; border: 1px solid #00A97240; white-space: nowrap;">Domain knowledge</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; border-bottom: 1px solid #EEEDE9;">A missing <code>OnlineSecurity</code> may <em>mean</em> "not subscribed"; if the data definition confirms it, fill with "No", not the mode</td></tr><tr><td style="padding: 12px 16px; border-left: 5px solid #2272B4; vertical-align: top; "><span style="display: inline-block; padding: 4px 12px; border-radius: 999px; background: #EBF3FA; color: #2272B4; font-weight: 700; border: 1px solid #2272B440; white-space: nowrap;">Where you fit it</span></td><td style="padding: 12px 16px; color: #0b2026; line-height: 1.55; vertical-align: top; ">Fit on the training set only, inside a Pipeline, and reuse the fitted pipeline for test and inference data</td></tr></tbody></table></div>

### E2. Mark Imputed Values

When the fact that a value was missing may carry signal, add an **indicator column** (created before imputing, as in C2). It preserves the fact that a value was originally missing. Use indicators selectively and check whether they help; you do not need one for every imputed column.

<div style="border: 1px solid #DCE0E2; border-radius: 8px; overflow: hidden; max-width: 860px; margin: 8px 0 16px; box-shadow: 0 2px 8px rgba(27,49,57,0.06);"><table style="border-collapse: collapse; width: 100%; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; font-size: 14px;"><thead style="background: #EEEDE9;"><tr><th style="padding: 8px 14px; text-align: left; font-weight: 600; color: #0b2026; border-bottom: 1px solid #DCE0E2;">ID</th><th style="padding: 8px 14px; text-align: left; font-weight: 600; color: #0b2026; border-bottom: 1px solid #DCE0E2;">Name</th><th style="padding: 8px 14px; text-align: left; font-weight: 600; color: #0b2026; border-bottom: 1px solid #DCE0E2;">Age</th><th style="padding: 8px 14px; text-align: left; font-weight: 600; color: #0b2026; border-bottom: 1px solid #DCE0E2;">Age_was_missing</th></tr></thead><tbody><tr><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">1</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">Alice</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">25.0</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">0</td></tr><tr><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">2</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">Bob</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">30.0</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">0</td></tr><tr><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">3</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">Charlie</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;"><strong style="color: #FF5F46;">28.0</strong></td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;"><strong style="color: #FF5F46;">1</strong></td></tr><tr><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">4</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">David</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">28.0</td><td style="padding: 8px 14px; border-bottom: 1px solid #EEEDE9; color: #0b2026;">0</td></tr></tbody></table></div>

Indicator columns are useful for:

- **Model evaluation** — segment performance by imputed vs. observed values
- **Transparency** — auditors can distinguish observed values from filled-in ones

## F. Conclusion

In this lecture, you learned how to handle missing data on Databricks:

1. **Find missing values** with the Data Profile, `summary()`, Spark null counts, and Genie Code, and track them over time with Data profiling in Unity Catalog.
2. **Choose a strategy** for each column: drop, treat missing as a category, use a domain default, or impute.
3. **Impute with Spark ML's `Imputer`**, which learns mean, median, or mode values from the training set and applies them consistently to test and inference data.
4. **Build imputation into a Pipeline**, add missingness indicators where they help, and save the fitted pipeline to a Unity Catalog volume for reuse.
5. **Model-based imputation** (KNN or regression) is optional: it can capture relationships between columns, but it adds cost, and a regression prediction is a single best guess, not a measure of uncertainty.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://www.databricks.com/legal/privacynotice" target="_blank">Privacy Notice</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Community</a>